In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes datasets \
    sentence-transformers faiss-cpu rouge-score

import os, pickle, numpy as np, re, torch, random
from collections import Counter

random.seed(42); np.random.seed(42)

In [ ]:
# ── Cell 2: Mount Drive & Load Data ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE = 'your/base/path/here'

with open(f'{BASE}/corpus.pkl', 'rb') as f: corpus = pickle.load(f)
with open(f'{BASE}/queries.pkl', 'rb') as f: queries = pickle.load(f)
print(f"✅ {len(corpus)} passages | {len(queries)} queries")

In [ ]:
# ── Cell 3: FAISS + Model ──────────────────────────────────────────────────────
!pip install -q faiss-cpu
!pip install --upgrade bitsandbytes
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')
index    = faiss.read_index(f'{BASE}/faiss_index.bin')

USE_LLAMA = True
HF_TOKEN  = "your_huggingface_token_here"
MODEL_NAME = ("meta-llama/Llama-3.1-8B-Instruct" if USE_LLAMA
              else "microsoft/Phi-3-mini-4k-instruct")
token_arg = HF_TOKEN if USE_LLAMA else None

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token_arg)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                  device_map="auto", token=token_arg)
model.eval()
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print("✅ Model loaded")

In [ ]:
# ── Cell 4: Helpers (same as NB1) ─────────────────────────────────────────────
def retrieve(question, top_k=5):
    q = embedder.encode([question], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q)
    sc, idx = index.search(q, top_k)
    return [{'passage': corpus[i], 'score': float(s)} for i, s in zip(idx[0], sc[0])]

def _fmt(question, docs=None):
    if USE_LLAMA:
        sys = "You are a helpful assistant. Use the documents to answer accurately. If they help, use them; otherwise use your knowledge."
        if docs:
            ctx  = "\n\n".join(f"[Doc {i+1}]: {d['passage']['text']}" for i, d in enumerate(docs))
            user = f"Documents:\n{ctx}\n\nQuestion: {question}\n\nAnswer (1-2 sentences):"
        else:
            user = f"Question: {question}\n\nAnswer (1-2 sentences):"
        return (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{sys}\n"
                f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user}"
                f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n")
    else:
        ctx = ("\n\n".join(f"[Doc {i+1}]: {d['passage']['text']}" for i, d in enumerate(docs)) if docs else "")
        prefix = f"Documents:\n{ctx}\n\n" if docs else ""
        return f"<|user|>\n{prefix}Question: {question}\nAnswer briefly:<|end|>\n<|assistant|>\n"

def generate(prompt, max_new_tokens=80):
    inp = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=1.0, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def norm(s):
    s = re.sub(r'\b(a|an|the)\b', ' ', s.lower())
    return ' '.join(re.sub(r'[^\w\s]', '', s).split())

def em(pred, golds): return int(any(norm(g)==norm(pred) for g in golds))
def f1(pred, golds):
    pt = norm(pred).split(); best = 0
    for g in golds:
        gt = norm(g).split(); c = Counter(pt)&Counter(gt); nc = sum(c.values())
        if nc==0: continue
        pr=nc/len(pt) if pt else 0; rc=nc/len(gt) if gt else 0
        if pr+rc>0: best=max(best,2*pr*rc/(pr+rc))
    return best

def save_ckpt(d,p):
    with open(p,'wb') as f: pickle.dump(d,f)
def load_ckpt(p):
    if os.path.exists(p):
        with open(p,'rb') as f: return pickle.load(f)
    return []

In [ ]:
# ── Cell 5: Noise Injection Functions ─────────────────────────────────────────
def add_contradictions(text):
    """Flip numbers and swap antonym pairs."""
    def flip(m):
        n=int(m.group())
        return str(n*2 if 0<n<10000 else n-1)
    text = re.sub(r'\b\d+\b', flip, text)
    pairs = [('won','lost'),('first','last'),('north','south'),('east','west'),
             ('before','after'),('largest','smallest'),('founded','dissolved'),
             ('increased','decreased'),('early','late'),('born','died')]
    for a, b in pairs:
        if re.search(r'\b'+a+r'\b', text, re.I):
            text = re.sub(r'\b'+a+r'\b', b, text, flags=re.IGNORECASE)
            break  # one swap per passage to avoid over-corruption
    return text

def shift_dates(text, years=10):
    def sub(m):
        y=int(m.group())
        return str(y-years) if 1800<=y<=2024 else m.group()
    return re.sub(r'\b(1[89]\d{2}|20[012]\d)\b', sub, text)

def truncate_partial(text):
    words = text.split()
    return ' '.join(words[:max(15, len(words)//2)])

def inject_noise(docs, noise_type, noise_level):
    """
    Replace `noise_level` fraction of docs with noisy versions.
    noise_level: 0.25 | 0.5 | 0.75
    """
    docs = [dict(d) for d in docs]
    n_noisy = max(1, round(len(docs) * noise_level))
    for idx in random.sample(range(len(docs)), min(n_noisy, len(docs))):
        p = dict(docs[idx]['passage'])
        if noise_type == 'irrelevant':
            docs[idx]['passage'] = corpus[random.randint(0, len(corpus)-1)]
        elif noise_type == 'contradictory':
            p['text'] = add_contradictions(p['text']); docs[idx]['passage'] = p
        elif noise_type == 'outdated':
            p['text'] = shift_dates(p['text']); docs[idx]['passage'] = p
        elif noise_type == 'partial':
            p['text'] = truncate_partial(p['text']); docs[idx]['passage'] = p
        elif noise_type == 'mixed':
            sub = random.choice(['irrelevant','contradictory','outdated','partial'])
            docs[idx:idx+1] = inject_noise([docs[idx]], sub, 1.0)
    return docs

In [ ]:
# ── Cell 6: Experiment Config ──────────────────────────────────────────────────
# Set QUICK_MODE=True for class deadline (3 configs, ~40 min)
QUICK_MODE = False

if QUICK_MODE:
    NOISE_TYPES  = ['irrelevant', 'contradictory', 'outdated']
    NOISE_LEVELS = [0.5]
    print("⚡ QUICK MODE: 3 configs (~40 min)")
else:
    NOISE_TYPES  = ['irrelevant', 'contradictory', 'outdated', 'partial', 'mixed']
    NOISE_LEVELS = [0.25, 0.5, 0.75]
    print(f"🔬 FULL MODE: {len(NOISE_TYPES)*len(NOISE_LEVELS)} configs (~2-3 hrs)")


In [ ]:
# ── Cell 7: Run All Noise Experiments ─────────────────────────────────────────
all_noisy = {}

for noise_type in NOISE_TYPES:
    for noise_level in NOISE_LEVELS:
        cfg   = f"{noise_type}_{int(noise_level*100)}"
        ckpt  = f"{BASE}/checkpoints/p2_{cfg}.pkl"
        final = f"{BASE}/results/p2/{cfg}_final.pkl"

        # Load if already fully done
        if os.path.exists(final):
            with open(final,'rb') as f: all_noisy[cfg] = pickle.load(f)
            print(f"✅ Skipping {cfg} (already done, F1={np.mean([r['f1'] for r in all_noisy[cfg]]):.3f})")
            continue

        results = load_ckpt(ckpt)
        start   = len(results)
        print(f"\n▶ {cfg}: resuming from {start}/{len(queries)}")

        for i, q in enumerate(queries[start:], start=start):
            docs      = retrieve(q['question'], top_k=5)
            noisy_docs = inject_noise(docs, noise_type, noise_level)
            ans       = generate(_fmt(q['question'], noisy_docs))
            results.append({
                'query_id': q['id'], 'question': q['question'],
                'answers': q['answers'], 'answer': ans,
                'noisy_docs': noisy_docs,
                'noise_type': noise_type, 'noise_level': noise_level,
                'em': em(ans, q['answers']), 'f1': f1(ans, q['answers'])
            })
            if (i+1) % 25 == 0 or i == len(queries)-1:
                save_ckpt(results, ckpt)
                print(f"  [{i+1}/{len(queries)}] F1={np.mean([r['f1'] for r in results]):.3f} ✅")

        save_ckpt(results, final)
        all_noisy[cfg] = results
        print(f"✅ {cfg} DONE | F1={np.mean([r['f1'] for r in results]):.3f}")

with open(f'{BASE}/results/p2/all_noisy.pkl','wb') as f: pickle.dump(all_noisy, f)


In [ ]:
# ── Cell 8: Summary ────────────────────────────────────────────────────────────
print("\n" + "="*45)
print(f"{'Config':<28} {'EM':>6} {'F1':>6}")
print("="*45)
for cfg, res in sorted(all_noisy.items()):
    print(f"{cfg:<28} {np.mean([r['em'] for r in res]):>6.3f} {np.mean([r['f1'] for r in res]):>6.3f}")
print("="*45)
print("\n👉 Next: Open Notebook 3 (GPU runtime)")